# Decision / SystemOne models - validation

**Purpose:** validate native typed decisions through Unified AI API, without translating provider payloads.

| Scenario | Route / evidence |
|---|---|
| OpenAI | POST `/unified-ai/v1/decisions` and `/unified-ai/openai/v1/decisions`; predicate, choice, score, multiple questions, image input |
| Microsoft | POST `/unified-ai/providers/microsoft/v1/systemone`; noul, choice, score, text and JSON state |
| Governance | Missing/invalid keys, missing model, RBAC, method/path rejection, wrong provider, isolated request rate/quota |
| Compatibility | Lossless backend additions, existing aliases/settings, optional legacy chat smoke test |

**Expected outcomes:** exact routing headers, usable typed answers, actual usage, explicit provider skips and a strict summary. No latency or prediction threshold is assumed. Refusals are reported as failed classification coverage, not fabricated answers.

**Azure prerequisites:** an existing gateway with the common Decision routing fragments upgraded, live model registrations and APIM permissions. Run All discovers `gpt-6-luna` and `Decision-1` automatically on compatible gateway backends. No capability override, local file or complete onboarding source is required when they are registered. No model deployments are created here.

**Safety and cost:** Run All creates access contracts and billable inference requests. With the target models registered it never reconfigures shared backends, pools or routing fragments, regardless of stale azd backend snapshots. Only an explicitly supplied source/additions for a missing provider enables optional lossless onboarding. Cleanup is off and only removes run-owned products/subscriptions. Outputs are credential-redacted and must be cleared before commit.

**Governance limitation:** APIM does not currently document either Decision schema for built-in token limits/metrics. This notebook deliberately uses request rate/quota policies and reports native usage. It does not certify existing token-only products for Decision capacity enforcement. Key Vault and Foundry connection creation are disabled for this throwaway validation contract.

See [staged upgrade](../bicep/infra/apim-gateway-upgrade/README.md#decision--systemone-release-upgrade), [backend onboarding](../bicep/infra/llm-backend-onboarding/README.md#decision-models), [OpenAI documentation](https://learn.microsoft.com/azure/foundry/openai/how-to/decisions), and [Microsoft documentation](https://learn.microsoft.com/azure/foundry/foundry-models/how-to/use-foundry-models-microsoft-decision).


## 0. Initialize notebook variables
**Input:** active azd environment or gateway overrides. Models come from the live gateway, not azd's saved backend inventory. Known targets are detected case-insensitively; alternate deployment names can use capability metadata or optional overrides.

**Output:** editable configuration only; no discovery or deployment. Leave source/additions empty for normal Run All. They are consulted only if an enabled provider is missing from live discovery. Independent APIM services are never copied into the LLM contract.


In [2]:
from pathlib import Path
import sys
repo_root = next((p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    if (p / 'azure.yaml').is_file() and (p / 'shared' / 'utils.py').is_file()), None)
if repo_root is None:
    raise RuntimeError('Open the repository before running this notebook.')
sys.path.insert(0, str(repo_root / 'shared'))
import utils
with utils.notebook_step('Configure Decision validation', inputs={}, expected='Editable configuration; no cloud operations.', namespace=globals()):
    import base64, copy, importlib, json, struct, time, uuid, zlib
    import decisiontools
    importlib.reload(decisiontools)
    from apimtools import APIMClientTool
    from validationtools import ValidationContext, ValidationRun, RequestTransport, valid_chat
    from notebookdefaults import backend_source, source_catalog
    from consumerhelpers import complete_backend_parameters, review_backend_artifact
    from contracttools import prepare_artifact, scoped_deployment_name
    from scenariotools import access_parameters, contract_identity, cleanup_products, save_manifest
    from decisiontools import ROUTES, DEFAULT_DECISION_MODELS, live_decision_catalog, build_product_policy_xml, decision_payload, decision_inference_errors, preserve_live_contract, validate_llm_backend_inventory

    # GOVERNANCE HUB
    init_from_azd = True
    environment = None
    overrides = {}
    source_parameters_file = None
    backend_additions = []
    run_backend_onboarding = True
    backend_configuration_complete = True
    run_id = uuid.uuid4().hex[:12]
    cleanup_delete_contract = False
    # API VERSION: both new surfaces use version-in-path; no dated query version.
    test_dated_api_version = '2024-02-01'
    # MODEL: discover live targets automatically; overrides are optional.
    operation_overrides = None
    skipped_models = None
    test_openai = True
    test_microsoft = True
    openai_decision_override = None
    microsoft_decision_override = None
    requests_per_minute = 600
    request_quota = 10000
    quota_period_seconds = 86400
    test_request_limits = True
    # KEY VAULT / MICROSOFT FOUNDRY: not used by this throwaway validation contract.
    use_keyvault = False
    use_foundry = False


## 1. Verify Azure CLI and initialize APIM
**Input:** configuration above and live gateway backend/model discovery. azd supplies gateway coordinates; its saved backend list does not determine the scenario's model inventory.

**Output:** verified context, discovered API and selected live providers. Registered targets bypass backend-source checks. Missing providers are recorded as SKIP after optional onboarding; configured request failures remain FAIL.


In [3]:
with utils.notebook_step('Initialize Decision context', inputs={'environment': environment, 'overrides': overrides}, expected='Verified subscription, preserved source, eligible deployments.', namespace=globals()):
    run = ValidationRun('decision-models')
    context = ValidationContext.load(overrides=overrides, environment=environment, from_azd=init_from_azd,
        operation_overrides=operation_overrides, skipped_models=skipped_models)
    utils.require_inputs({key: context.config.get(key) for key in ('subscription_id', 'resource_group', 'location', 'apim_name')})
    apimClientTool = APIMClientTool(context.config['resource_group'], context.config['apim_name'])
    apimClientTool.initialize_context(context.account, enumerate_subscriptions=False)
    unified_api = apimClientTool.discover_api_exact('unified-ai')
    gateway_url = apimClientTool.apim_resource_gateway_url.rstrip('/')
    transport = RequestTransport(timeout=120)
    def fragment(name):
        return apimClientTool.client.policy_fragment.get(context.config['resource_group'], apimClientTool.apim_resource_name, name).value
    if 'is-decision-request' not in fragment('request-processor') or 'systemone' not in fragment('path-builder'):
        raise ValueError('Upgrade common Decision fragments first; see the staged gateway-upgrade guide.')
    live_result = transport.request('GET', gateway_url + '/version/backend-contract')
    if not live_result.check():
        raise ValueError('Cannot establish the complete live backend contract before shared routing changes.')
    live_contract = live_result.json_body
    live_backends, live_pools = apimClientTool.get_backends()
    live_catalog_config = {**context.config, 'capabilities': operation_overrides or {}}
    context.catalog = live_decision_catalog(live_contract, live_backends, live_catalog_config)
    provider_options = (
        ('decisions', test_openai, openai_decision_override, {'ai-foundry', 'azure-openai'}),
        ('systemone', test_microsoft, microsoft_decision_override, {'azure-decision'}),
    )
    def select_live_models():
        choices = {}
        for operation, enabled, override, types in provider_options:
            eligible = [name for name in context.catalog.models_for(operation)
                if any(e.name == name and e.backend_type in types for e in context.catalog.entries)]
            if not enabled:
                continue
            preferred = override or DEFAULT_DECISION_MODELS[operation]
            match = next((name for name in eligible if name.casefold() == preferred.casefold()), None)
            if match or (eligible and override is None):
                choices[operation] = match or eligible[0]
        return choices
    selected = select_live_models()
    source = desired = None
    live_ids = {item['backendId'] for item in live_contract['backends']}
    run_dir = repo_root / 'validation' / '.runs' / 'decision-models' / run_id
    owned_products = set()
    preexisting_products = {p.name for p in apimClientTool.client.product.list_by_service(
        context.config['resource_group'], apimClientTool.apim_resource_name)}
    utils.format_output('Selected Decision providers', selected, status='info', part='RESULT')


## 2. Generate and preview complete backend onboarding
**Input:** selected live providers. If enabled targets are registered, this step is a no-op and proceeds toward access-contract creation.

**Output:** an explicit onboarding skip for existing targets. Only a missing provider plus explicitly configured source/additions generates an artifact; that path preserves all predecessor records, aliases, auth references and global settings.


In [4]:
with utils.notebook_step('Preview Decision backend contract', inputs={'selected': selected}, expected='Complete onboarding artifact or an explicit no-op.', namespace=globals()):
    onboarding_artifact = None
    missing_providers = [operation for operation, enabled, _, _ in provider_options if enabled and operation not in selected]
    onboarding_needed = bool(missing_providers and (source_parameters_file or backend_additions))
    if onboarding_needed:
        if not run_backend_onboarding:
            raise ValueError('Decision backend/routing update is required but onboarding is disabled.')
        source = backend_source(context, apimClientTool, source_parameters_file, overrides.get('backend_template_options'))
        desired = complete_backend_parameters(source, backend_additions, confirmed_complete=backend_configuration_complete)
        desired = preserve_live_contract(desired, live_contract, verified_apim_resource_id=apimClientTool.apim_service_id)
        live_ids = validate_llm_backend_inventory(desired, live_contract, live_backends, live_pools)
        onboarding_artifact = prepare_artifact(repo_root / 'bicep' / 'infra' / 'llm-backend-onboarding' / 'main.bicep', desired, run_dir / 'backends')
        utils.format_output('Backend contract preview', {'parameters': onboarding_artifact.preview()}, status='info', part='RESULT')
    else:
        utils.print_info('Backend onboarding skipped: using live gateway registrations without changing backends, pools or routing fragments.')


## 3. Deploy backend changes when needed
**Input:** actual edited onboarding artifact and complete predecessor source. Deployment is enabled by default, matching the image notebook.

**Output:** updated shared backend contract and refreshed APIM state. Edited artifacts may add new backends/models only through the reviewed source; hidden predecessor/global changes are rejected. Shared additions are not deleted by cleanup.


In [5]:
with utils.notebook_step('Deploy Decision backend contract', inputs={'deployment_needed': onboarding_artifact is not None}, expected='Verified effective backend contract; APIM state refreshed.', namespace=globals()):
    if onboarding_artifact is not None:
        effective = review_backend_artifact(onboarding_artifact, desired, desired['apim'], live_ids)
        preserve_live_contract(effective, live_contract, verified_apim_resource_id=apimClientTool.apim_service_id)
        latest = transport.request('GET', gateway_url + '/version/backend-contract')
        if not latest.check() or any(latest.json_body.get(key) != live_contract.get(key) for key in ('backends', 'modelAliases', 'circuitBreaker', 'sessionAffinity')):
            raise ValueError('Shared backend contract changed after preview; restart initialization and review.')
        preserve_live_contract(effective, latest.json_body, verified_apim_resource_id=apimClientTool.apim_service_id)
        current_backends, current_pools = apimClientTool.get_backends()
        validate_llm_backend_inventory(effective, latest.json_body, current_backends, current_pools)
        onboarding_artifact.deploy(context.config['location'], scoped_deployment_name('validation-decision-backends', apimClientTool.apim_service_id))
        apimClientTool.initialize_context(context.account, enumerate_subscriptions=False)
        unified_api = apimClientTool.discover_api_exact('unified-ai')
        refreshed = transport.request('GET', gateway_url + '/version/backend-contract')
        if not refreshed.check() or refreshed.json_body.get('backends') != effective['llmBackendConfig']:
            raise ValueError('Deployed backend contract differs from the reviewed artifact.')
        preserve_live_contract(effective, refreshed.json_body)
        live_contract = refreshed.json_body
        live_backends, live_pools = apimClientTool.get_backends()
        context.catalog = live_decision_catalog(live_contract, live_backends, live_catalog_config)
        selected = select_live_models()
    for operation, enabled, override, _ in provider_options:
        if not enabled:
            run.skip(operation, 'Disabled explicitly.')
        elif operation not in selected:
            if override is not None:
                raise ValueError(f'{operation}: override is not registered on a compatible live backend.')
            run.skip(operation, 'Target model is not registered on a compatible live gateway backend.')


## 4. Generate the Decision access contract
**Input:** selected deployment names and request limits. The LLM product maps only to the discovered Unified AI API. Native usage is reported; built-in token accounting is not claimed.

**Output:** dynamic product XML and a run-owned `.bicepparam` preview. No copied static token policy. Key Vault and Foundry connections are not enabled.


In [6]:
with utils.notebook_step('Preview Decision access contract', inputs={'selected': selected}, expected='Dynamic request-limited contract artifact.', namespace=globals()):
    def contract_config(suffix='', rate=requests_per_minute, quota=request_quota):
        return {'name': 'validation-decisions' + suffix, 'business_unit': 'Testing',
            'use_case_name': 'Decisions' + suffix, 'environment': 'DEV',
            'allowed_models': list(dict.fromkeys(selected.values())), 'use_keyvault': False,
            'apikey_secret': 'DECISION-KEY', 'endpoint_secret': 'DECISION-ENDPOINT',
            'requests_per_minute': rate, 'request_quota': quota,
            'quota_period_seconds': quota_period_seconds, 'counter_scope': run_id}
    def access_artifact(contract, directory):
        policy = build_product_policy_xml(contract)
        parameters = access_parameters(context, apimClientTool, contract, policy, api_names=[unified_api['name']])
        return prepare_artifact(repo_root / 'bicep' / 'infra' / 'citadel-access-contracts' / 'main.bicep',
            parameters, directory, policies={'ai-product-policy.xml': policy})
    main_contract = contract_config() if selected else None
    main_artifact = access_artifact(main_contract, run_dir / 'access') if selected else None
    if main_artifact:
        utils.format_output('Access contract preview', {'parameters': main_artifact.preview()}, status='info', part='RESULT')


## 5. Deploy the access contract
**Input:** the previewed artifact. Only its subscription key is retrieved; credentials are never saved in the run manifest.

**Output:** isolated product/subscription, refreshed APIM client, headers, and request helper. Every request is recorded with native response evidence.


In [7]:
with utils.notebook_step('Deploy Decision access contract', inputs={'selected': selected}, expected='Owned contract and exact subscription key.', namespace=globals()):
    def deploy_access(artifact, contract):
        outputs = artifact.deploy(context.config['location'], scoped_deployment_name(contract['name'], apimClientTool.apim_service_id), required_outputs=('products', 'subscriptions'))
        _, subscription = contract_identity(outputs, preexisting_products, owned_products)
        apimClientTool.initialize_context(context.account, enumerate_subscriptions=False)
        return apimClientTool.get_subscription_key(subscription)
    headers = {'api-key': deploy_access(main_artifact, main_contract)} if selected else {}
    base_url = unified_api['url'].rstrip('/')
    def invoke(operation, payload, route=None, request_headers=None, *, require_model=True, **options):
        if require_model and (not isinstance(payload, dict) or not isinstance(payload.get('model'), str) or not payload['model'].strip()):
            raise ValueError('Decision JSON request must include a nonempty model deployment name.')
        return transport.request('POST', base_url + (route or ROUTES[operation][0]),
            headers=headers if request_headers is None else request_headers, json=payload, **options)
    def error_body(body):
        return isinstance(body, dict) and (isinstance(body.get('error'), (dict, str)) and bool(body['error']) or isinstance(body.get('message'), str) and bool(body['message'].strip()))


## 6. Native Decision inference: test plan and request samples

**Purpose:** prove that the Unified AI API forwards native typed decisions to the correct provider. All positive requests contain the selected deployment name in **`model`**. OpenAI and Microsoft payloads are different and are not translated.

### Tests performed by this step

| Test | OpenAI Decisions | Microsoft SystemOne | What it proves |
|---|---|---|---|
| Binary decision | `predicate` question `outage` | `noul` question `outage` | The provider returns a finite probability between 0 and 1. |
| Classification | `choice` question `team` | `choice` question `team` | The selected team is one of the supplied options, with per-option probabilities. |
| Grading | `score` question `severity` | `score` question `severity` | The score lies within the three-level scale (0 through 2). |
| Multiple questions | Ordered question/answer arrays | Named question/answer objects | All three independent questions are answered in a single request; names, types and option counts are checked. |
| Gateway route aliases | Both OpenAI entry paths below | The Microsoft provider path below | Each path selects the correct API type, deployment, compatible backend pool and final backend path. These are URL aliases, not model-alias/failover tests. |
| Fresh and normal metadata | One cache-bypass request, then one normal request per path | Same sequence | Both requests retain the same JSON `model`; the second also supplies a dated `api-version` to test gateway removal. This is not a request without a model. |
| Alternate input | Text plus a deterministic in-memory PNG | Structured JSON `state` | Native image input or JSON-state input produces usable typed answers without external files or image services. |
| Usage and refusal handling | Native input/output/total usage | Native input/output usage; total may be absent | Actual usage is present. Refusals, malformed answers and failed HTTP responses do not become successful classification evidence. |

### Endpoints and backend types

| Backend type | POST endpoint exposed by the gateway | Path forwarded to the backend |
|---|---|---|
| `ai-foundry` or `azure-openai` | `https://<gateway>/unified-ai/v1/decisions` | `/openai/v1/decisions` |
| `ai-foundry` or `azure-openai` (URL alias) | `https://<gateway>/unified-ai/openai/v1/decisions` | `/openai/v1/decisions` |
| `azure-decision` | `https://<gateway>/unified-ai/providers/microsoft/v1/systemone` | `/providers/microsoft/v1/systemone` |

All requests use `Content-Type: application/json` and the access-contract subscription key in `api-key`. The run prints the actual gateway URL, selected model and generated JSON sample for each provider; credentials are not included in the sample.

### OpenAI request sample - classify and grade a service outage

POST to either OpenAI endpoint above. `model` is the deployment name discovered for OpenAI (normally `gpt-6-luna`). `input` is the evidence shared by the three questions.

```json
{
  "model": "gpt-6-luna",
  "input": "The checkout API returns 500 for every customer.",
  "questions": [
    {"name": "outage", "type": "predicate", "instructions": "Is the service failing?"},
    {"name": "team", "type": "choice", "instructions": "Which team should handle this?",
     "choices": [{"value": "engineering", "description": "Errors and outages"}, {"value": "billing", "description": "Invoices"}]},
    {"name": "severity", "type": "score", "instructions": "How severe is the issue?",
     "levels": [{"label": "Cosmetic", "description": "No loss of functionality."}, {"label": "Degraded", "description": "Some requests fail but a workaround exists."}, {"label": "Blocked", "description": "All requests fail with no workaround."}]}
  ]
}
```

The image test keeps `model` and `questions` unchanged and replaces `input` with a user message containing `input_text` plus an `input_image` data URL.

### Microsoft request sample - the same decisions with native SystemOne

POST to the Microsoft endpoint above. `model` is the discovered Microsoft deployment (normally `Decision-1`). `state` supplies the evidence and `questions` is a keyed object, not an array.

```json
{
  "model": "Decision-1",
  "state": "The checkout API returns 500 for every customer.",
  "questions": {
    "outage": {"type": "noul", "instructions": "Is the service failing?"},
    "team": {"type": "choice", "instructions": "Which team should handle this?",
             "criteria": {"engineering": "Errors and outages", "billing": "Invoices"}},
    "severity": {"type": "score", "instructions": "How severe is the issue?",
                 "criteria": ["Cosmetic", "Degraded", "Blocked"]}
  }
}
```

The JSON-state test keeps `model` and `questions` unchanged and replaces `state` with `{"ticket": "The checkout API returns 500 for every customer."}`. Microsoft returns the binary value in `answers.outage.noul`.

**Input:** the live-discovered `selected` deployments, their native JSON payloads and the access-contract key.

**Output:** per-test PASS/FAIL, request-model evidence, native answers/usage and routing headers. HTTP errors show the actual provider message and region before any success-schema checks. A response without a `model` field is not evidence that the request omitted it. East US 2 currently does not support OpenAI Decisions; such a pool member can return HTTP 400 **unsupported operation** even when `model` is present. See [provider availability](https://learn.microsoft.com/azure/foundry/openai/how-to/decisions). This step does not modify backend pools or mask provider failures. Authentication, missing-model, RBAC and request-limit negative tests run in later steps.


In [8]:
with utils.notebook_step('Validate native Decision inference', inputs={'selected': selected}, expected='Typed answers, real usage and exact routing.', namespace=globals()):
    provider_success = set()
    for operation, model in selected.items():
        utils.format_output('Decision request sample', {'provider': operation,
            'POST endpoints': [base_url + route for route in ROUTES[operation]],
            'JSON body': decision_payload(operation, model)}, status='info', part='RESULT')
        for route in ROUTES[operation]:
            payload = decision_payload(operation, model)
            prime = invoke(operation, payload, route, request_headers={**headers, 'UAIG-Config-Cache-Bypass': 'true'})
            result = invoke(operation, payload, route, params={'api-version': test_dated_api_version})
            for label, response in (('cache-refresh', prime), ('ordinary', result)):
                errors = decision_inference_errors(response, payload, operation)
                expected_path = '/providers/microsoft/v1/systemone' if operation == 'systemone' else '/openai/v1/decisions'
                expected_type = 'systemone' if operation == 'systemone' else 'decisions-openai-v1' if route.startswith('/openai/') else 'decisions'
                compatible = {'azure-decision'} if operation == 'systemone' else {'ai-foundry', 'azure-openai'}
                expected_pools = {p['poolName'] for p in live_contract['pools'] if p['poolType'] in compatible and model in p['supportedModels']}
                routing_ok = (response.headers.get('UAIG-Final-Path') == expected_path
                    and response.headers.get('UAIG-API-Type') == expected_type
                    and response.headers.get('UAIG-Model-Id') == model
                    and response.headers.get('UAIG-Backend') in expected_pools)
                passed = response.check() and not errors and routing_ok
                run.check(f'{route}:{label}', passed, 'Native typed answers and exact routing',
                    {'request_model': payload['model'], 'errors': errors, 'status': response.status_code, 'region': response.headers.get('x-ms-region'),
                     'backend_message': response.json_body.get('message') if isinstance(response.json_body, dict) else None,
                     'headers': response.gateway_headers, 'usage': response.usage})
                if passed:
                    provider_success.add(operation)
        if operation == 'systemone':
            extra = decision_payload(operation, model, json_state=True)
        else:
            def png_chunk(kind, data):
                return struct.pack('!I', len(data)) + kind + data + struct.pack('!I', zlib.crc32(kind + data) & 0xffffffff)
            png = b'\x89PNG\r\n\x1a\n' + png_chunk(b'IHDR', struct.pack('!2I5B', 32, 32, 8, 2, 0, 0, 0))
            png += png_chunk(b'IDAT', zlib.compress((b'\0' + b'\xff\0\0' * 32) * 32)) + png_chunk(b'IEND', b'')
            extra = decision_payload(operation, model, image_url='data:image/png;base64,' + base64.b64encode(png).decode('ascii'))
        response = invoke(operation, extra)
        errors = decision_inference_errors(response, extra, operation)
        run.check(operation + ':image-or-json-state', response.check() and not errors, 'Valid typed answers',
            {'request_model': extra['model'], 'errors': errors, 'status': response.status_code, 'usage': response.usage})


## 7. Authentication, RBAC and unsupported surfaces
**Input:** selected native payloads and actual inventory. RBAC uses a genuinely configured ungranted model when available; wrong-surface tests require both distinct deployments.

**Output:** checked HTTP status plus an error body. A configured provider's error is not reclassified as an availability skip. No unsupported GET lifecycle or streaming API is invented.


In [9]:
with utils.notebook_step('Validate Decision governance', inputs={'selected': selected}, expected='Explicit auth/model/path/method errors.', namespace=globals()):
    for operation, model in selected.items():
        payload = decision_payload(operation, model)
        for label, auth in (('missing-key', {}), ('invalid-key', {'api-key': 'invalid-validation-key'})):
            response = invoke(operation, payload, request_headers=auth)
            run.check(operation + ':' + label, response.check(401, error_body), '401 error', response.status_code)
        missing = dict(payload)
        missing.pop('model')
        response = invoke(operation, missing, require_model=False)
        run.check(operation + ':missing-model', response.check(400, error_body), '400 error', response.status_code)
        invalid = {'model': model, 'questions': None}
        response = invoke(operation, invalid)
        run.check(operation + ':invalid-native-payload', response.check((400, 422), error_body), 'Provider validation error', response.status_code)
        for route in ROUTES[operation]:
            response = transport.request('GET', base_url + route, headers=headers)
            run.check(route + ':method', response.check(405, error_body) and response.headers.get('Allow') == 'POST', '405 Allow: POST', response.status_code)
            response = invoke(operation, payload, route + '-extra')
            run.check(route + ':prefix-collision', response.check(403, error_body), '403 PathNotAllowed', response.status_code)
        candidates = [name for name in context.catalog.models_for(operation) if name not in selected.values()]
        if candidates:
            response = invoke(operation, decision_payload(operation, candidates[0]))
            run.check(operation + ':rbac', response.check(403, error_body), '403 for configured ungranted model', response.status_code)
        else:
            run.skip(operation + ':rbac', 'No configured ungranted Decision deployment is available.')
    if len(selected) == 2 and len(set(selected.values())) == 2:
        for operation, other in (('decisions', 'systemone'), ('systemone', 'decisions')):
            other_model = selected[other]
            compatible_types = {'ai-foundry', 'azure-openai'} if operation == 'decisions' else {'azure-decision'}
            if any(e.name == other_model and e.backend_type in compatible_types for e in context.catalog.entries):
                run.skip(operation + ':wrong-surface', 'Other deployment is also registered on this compatible surface.')
            else:
                response = invoke(operation, decision_payload(operation, other_model))
                run.check(operation + ':wrong-surface', response.check(400, error_body), '400 incompatible model/pool', response.status_code)
    else:
        run.skip('wrong-surface', 'Requires two distinct configured provider deployments.')


## 8. Isolated request limits and optional legacy smoke
**Input:** a successful selected provider. Each low-limit probe creates a separate run-owned product so throttling cannot poison the main contract. Up to four attempts per probe; no unbounded stress loop.

**Output:** actual threshold-crossing evidence for request rate (429) and quota (403). Optional legacy chat uses the existing chat capability and a separately generated request-governed contract; it never sends chat to Microsoft SystemOne.


In [10]:
with utils.notebook_step('Validate bounded limits and compatibility', inputs={'test_request_limits': test_request_limits}, expected='Observed request-limit statuses; optional legacy chat evidence.', namespace=globals()):
    if test_request_limits and provider_success:
        operation = next(op for op in ('systemone', 'decisions') if op in selected and op in provider_success)
        payload = decision_payload(operation, selected[operation])
        for suffix, rate, quota, expected in (('Rate', 1, 1000, 429), ('Quota', 600, 1, 403)):
            contract = contract_config(suffix, rate, quota)
            artifact = access_artifact(contract, run_dir / suffix.lower())
            utils.format_output('Isolated limit preview', {'parameters': artifact.preview()}, status='info', part='RESULT')
            key = deploy_access(artifact, contract)
            initial = invoke(operation, payload, request_headers={'api-key': key})
            run.check(suffix + ':initial', initial.check(), 'First request succeeds', initial.status_code)
            response = initial
            for attempt in range(4):
                response = invoke(operation, payload, request_headers={'api-key': key})
                if response.status_code == expected:
                    break
                if response.status_code != 200:
                    break
                time.sleep(0.25)
            run.check(suffix + ':threshold', response.check(expected, error_body), f'{expected} after one permitted request', response.status_code)
    else:
        run.skip('request-limits', 'Disabled explicitly or no successful provider to exercise.')
    chat_candidates = [name for name in context.catalog.models_for('chat') if any(e.name == name and e.backend_type in {'ai-foundry', 'azure-openai'} for e in context.catalog.entries)]
    if selected and chat_candidates:
        contract = contract_config('Legacy')
        contract['allowed_models'] = [chat_candidates[0]]
        artifact = access_artifact(contract, run_dir / 'legacy')
        key = deploy_access(artifact, contract)
        response = transport.request('POST', base_url + '/v1/chat/completions', headers={'api-key': key},
            json={'model': chat_candidates[0], 'messages': [{'role': 'user', 'content': 'Reply with a short greeting.'}]})
        run.check('legacy-chat', response.check(body_check=valid_chat), 'Usable legacy chat completion', response.status_code)
    else:
        run.skip('legacy-chat', 'No eligible configured OpenAI-compatible chat deployment.')


## Results summary
**Input:** accumulated real checks and skips. Provider coverage is separate from authentication-only checks.

**Output:** strict verdict and sanitized manifest. Partial provider coverage is explicitly reported; zero successful providers cannot pass. Any failed configured request causes the summary to raise.


In [11]:
with utils.notebook_step('Summarize Decision coverage', inputs={'selected': selected}, expected='Strict verdict; no false both-provider success.', namespace=globals()):
    utils.format_output('Provider coverage', {'selected': selected, 'successful': sorted(provider_success),
        'coverage': 'both providers' if len(provider_success) == 2 else 'partial' if provider_success else 'none'},
        status='info' if len(provider_success) == 2 else 'WARNING', part='RESULT')
    if not provider_success:
        run.blocked('provider-coverage', 'No provider completed valid typed inference.')
    run_dir.mkdir(parents=True, exist_ok=True)
    save_manifest(run_dir, context, apimClientTool, owned_products)
    transport.close()
    summary = run.finish()


Sanitized run manifest: D:\Repos\ai-hub-gateway-solution-accelerator-1\validation\.runs\decision-models\2980c152c35f\manifest.json


case,status,required,expected,actual
/v1/decisions:cache-refresh,FAIL,True,Native typed answers and exact routing,"{""request_model"": ""gpt-6-luna"", ""errors"": [""Backend HTTP 400 (region: East US 2): The requested operation is unsupported. Request model: 'gpt-6-luna'.""], ""status"": 400, ""region"": ""East US 2"", ""backend_message"": null, ""headers"": {""UAIG-Auth-Type"": ""api-key"", ""UAIG-User-Id"": ""LLM-Testing-Decisions-DEV-SUB-01"", ""UAIG-Subscription"": ""LLM-Testing-Decisions-DEV-SUB-01"", ""UAIG-Model-Id"": ""gpt-6-luna"", ""UAIG-API-Type"": ""decisions"", ""UAIG-Processed-Path"": ""/v1/decisions"", ""UAIG-API-Version"": ""v1"", ""UAIG-Is-Streaming"": ""false"", ""UAIG-Backend"": ""gpt-6-luna-ai-foundry-backend-pool"", ""UAIG-Final-Path"": ""/openai/v1/decisions"", ""UAIG-Auth-Type-Backend"": ""native-backend-authorization"", ""UAIG-Cache-Operation"": ""cache-miss"", ""UAIG-Request-Id"": ""c9345c69-becd-47bc-9a46-8b3d96a04744"", ""UAIG-Gateway-Region"": ""Sweden Central""}, ""usage"": {}}"
/v1/decisions:ordinary,PASS,True,Native typed answers and exact routing,"{""request_model"": ""gpt-6-luna"", ""errors"": [], ""status"": 200, ""region"": ""Sweden Central"", ""backend_message"": null, ""headers"": {""UAIG-Auth-Type"": ""api-key"", ""UAIG-User-Id"": ""LLM-Testing-Decisions-DEV-SUB-01"", ""UAIG-Subscription"": ""LLM-Testing-Decisions-DEV-SUB-01"", ""UAIG-Model-Id"": ""gpt-6-luna"", ""UAIG-API-Type"": ""decisions"", ""UAIG-Processed-Path"": ""/v1/decisions"", ""UAIG-API-Version"": ""v1"", ""UAIG-Is-Streaming"": ""false"", ""UAIG-Backend"": ""gpt-6-luna-ai-foundry-backend-pool"", ""UAIG-Final-Path"": ""/openai/v1/decisions"", ""UAIG-Auth-Type-Backend"": ""native-backend-authorization"", ""UAIG-Cache-Operation"": ""cache-hit"", ""UAIG-Request-Id"": ""5eb289d5-62a2-44c9-b109-37aaaa552cfc"", ""UAIG-Gateway-Region"": ""Sweden Central""}, ""usage"": {""input_tokens"": 402, ""input_tokens_details"": ""[REDACTED]"", ""output_tokens"": 0, ""output_tokens_details"": ""[REDACTED]"", ""total_tokens"": 402}}"
/openai/v1/decisions:cache-refresh,FAIL,True,Native typed answers and exact routing,"{""request_model"": ""gpt-6-luna"", ""errors"": [""Backend HTTP 400 (region: East US 2): The requested operation is unsupported. Request model: 'gpt-6-luna'.""], ""status"": 400, ""region"": ""East US 2"", ""backend_message"": null, ""headers"": {""UAIG-Auth-Type"": ""api-key"", ""UAIG-User-Id"": ""LLM-Testing-Decisions-DEV-SUB-01"", ""UAIG-Subscription"": ""LLM-Testing-Decisions-DEV-SUB-01"", ""UAIG-Model-Id"": ""gpt-6-luna"", ""UAIG-API-Type"": ""decisions-openai-v1"", ""UAIG-Processed-Path"": ""/openai/v1/decisions"", ""UAIG-API-Version"": ""v1"", ""UAIG-Is-Streaming"": ""false"", ""UAIG-Backend"": ""gpt-6-luna-ai-foundry-backend-pool"", ""UAIG-Final-Path"": ""/openai/v1/decisions"", ""UAIG-Auth-Type-Backend"": ""native-backend-authorization"", ""UAIG-Cache-Operation"": ""cache-miss"", ""UAIG-Request-Id"": ""bd379fb1-097d-4138-a840-b47a48187a14"", ""UAIG-Gateway-Region"": ""Sweden Central""}, ""usage"": {}}"
/openai/v1/decisions:ordinary,PASS,True,Native typed answers and exact routing,"{""request_model"": ""gpt-6-luna"", ""errors"": [], ""status"": 200, ""region"": ""Sweden Central"", ""backend_message"": null, ""headers"": {""UAIG-Auth-Type"": ""api-key"", ""UAIG-User-Id"": ""LLM-Testing-Decisions-DEV-SUB-01"", ""UAIG-Subscription"": ""LLM-Testing-Decisions-DEV-SUB-01"", ""UAIG-Model-Id"": ""gpt-6-luna"", ""UAIG-API-Type"": ""decisions-openai-v1"", ""UAIG-Processed-Path"": ""/openai/v1/decisions"", ""UAIG-API-Version"": ""v1"", ""UAIG-Is-Streaming"": ""false"", ""UAIG-Backend"": ""gpt-6-luna-ai-foundry-backend-pool"", ""UAIG-Final-Path"": ""/openai/v1/decisions"", ""UAIG-Auth-Type-Backend"": ""native-backend-authorization"", ""UAIG-Cache-Operation"": ""cache-hit"", ""UAIG-Request-Id"": ""28733826-3ca1-4093-8dc0-de113dc8a558"", ""UAIG-Gateway-Region"": ""Sweden Central""}, ""usage"": {""input_tokens"": 402, ""input_tokens_details"

ValidationError: decision-models failed or has blocked required coverage

## Cleanup (optional)
**Input:** `cleanup_delete_contract=False` unless explicitly changed and positively recorded ownership.

**Output:** preview or deletion of this run's products/subscriptions only. Shared backends, model deployments, aliases, infrastructure and predecessor contracts are never deleted.


In [ ]:
with utils.notebook_step('Cleanup owned Decision products', inputs={'enabled': cleanup_delete_contract}, expected='Owned-resource-only cleanup; shared routing retained.', namespace=globals()):
    cleanup_products(apimClientTool, owned_products, enabled=cleanup_delete_contract)
    save_manifest(run_dir, context, apimClientTool, owned_products)
